In [5]:
# CELL 1: Imports and Environment Setup
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Configure Pandas to show all columns when we print dataframes
pd.set_option('display.max_columns', None)

print("✅ All modules imported successfully!")

✅ All modules imported successfully!


In [7]:
# CELL 2: Load the Datasets
# Assuming your notebook is in backend/ml/notebooks/ and data is in backend/ml/data/
DATA_DIR = "../data"

try:
    df_crop = pd.read_csv(os.path.join(DATA_DIR, "crop_yield.csv"))
    df_soil = pd.read_csv(os.path.join(DATA_DIR, "state_soil_data.csv"))
    df_weather = pd.read_csv(os.path.join(DATA_DIR, "state_weather_data_1997_2020.csv"))
    
    print("✅ All datasets loaded successfully!\n")
    
    print(f"Crop Yield Shape : {df_crop.shape}")
    print("Crop Yield Columns :", df_crop.columns.tolist(), "\n")
    
    print(f"Soil Data Shape  : {df_soil.shape}")
    print("Soil Data Columns  :", df_soil.columns.tolist(), "\n")
    
    print(f"Weather Shape    : {df_weather.shape}")
    print("Weather Columns    :", df_weather.columns.tolist())
    
except FileNotFoundError as e:
    print("❌ Error: Could not find the data files.")
    print(f"Details: {e}")
    print("Please check that your CSV files are exactly named and placed inside the 'backend/ml/data/' folder.")

✅ All datasets loaded successfully!

Crop Yield Shape : (19689, 9)
Crop Yield Columns : ['crop', 'year', 'season', 'state', 'area', 'production', 'fertilizer', 'pesticide', 'yield'] 

Soil Data Shape  : (30, 5)
Soil Data Columns  : ['state', 'N', 'P', 'K', 'pH'] 

Weather Shape    : (720, 5)
Weather Columns    : ['state', 'year', 'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent']


In [9]:
print("Crop columns   :", df_crop.columns.tolist())
print("Soil columns   :", df_soil.columns.tolist())
print("Weather columns:", df_weather.columns.tolist())

Crop columns   : ['crop', 'year', 'season', 'state', 'area', 'production', 'fertilizer', 'pesticide', 'yield']
Soil columns   : ['state', 'N', 'P', 'K', 'pH']
Weather columns: ['state', 'year', 'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent']


In [12]:
# CELL 3: Clean, Merge, and Filter for Punjab & Haryana

# 1. Rename your specific lowercase columns to the expected Title Case formats
df_crop.rename(columns={
    'crop': 'Crop', 'year': 'Year', 'state': 'State', 
    'area': 'Area', 'production': 'Production'
}, inplace=True)

df_soil.rename(columns={
    'state': 'State'
}, inplace=True)

df_weather.rename(columns={
    'state': 'State', 'year': 'Year', 
    'avg_temp_c': 'Temperature', 
    'total_rainfall_mm': 'Rainfall', 
    'avg_humidity_percent': 'Humidity'
}, inplace=True)

# 2. Standardize the text (strip hidden spaces and make it Title Case, e.g., 'punjab' -> 'Punjab')
for df in [df_crop, df_soil, df_weather]:
    df['State'] = df['State'].astype(str).str.strip().str.title()

df_crop['Crop'] = df_crop['Crop'].astype(str).str.strip().str.title()

# 3. Strict filter for Punjab and Haryana
target_states = ['Punjab', 'Haryana']
df_crop = df_crop[df_crop['State'].isin(target_states)].copy()
df_soil = df_soil[df_soil['State'].isin(target_states)].copy()
df_weather = df_weather[df_weather['State'].isin(target_states)].copy()

# 4. Merge: crop + weather (on State and Year)
df_merged = pd.merge(df_crop, df_weather, on=['State', 'Year'], how='left')

# 5. Merge: + soil data (on State)
df_merged = pd.merge(df_merged, df_soil, on=['State'], how='left')

# 6. Handle the Missing Soil Data (Injecting local averages)
soil_cols = ['N', 'P', 'K', 'pH']
for col in soil_cols:
    if col in df_merged.columns:
        # Fill missing values with the state average
        df_merged[col] = df_merged.groupby('State')[col].transform(lambda x: x.fillna(x.mean()))
        # If still missing, fill with global average
        df_merged[col] = df_merged[col].fillna(df_merged[col].mean())

print(f"✅ Merged Dataset Shape for Punjab & Haryana: {df_merged.shape}")
display(df_merged.head(10))

✅ Merged Dataset Shape for Punjab & Haryana: (1028, 16)


,Crop,Year,season,State,Area,Production,fertilizer,pesticide,yield,Temperature,Rainfall,Humidity,N,P,K,pH
0,Arhar/Tur,1997,Kharif,Punjab,11000.0,8200,1046870.0,3410.0,0.766154,23.39,942.38,46.2,150,50,40,8.0
1,Bajra,1997,Kharif,Punjab,8000.0,8000,761360.0,2480.0,1.000000,23.39,942.38,46.2,150,50,40,8.0
2,Barley,1997,Rabi,Punjab,37000.0,111000,3521290.0,11470.0,2.995455,23.39,942.38,46.2,150,50,40,8.0
3,Cotton(Lint),1997,Whole Year,Punjab,724000.0,937000,68903080.0,224440.0,1.324000,23.39,942.38,46.2,150,50,40,8.0
4,Gram,1997,Rabi,Punjab,13300.0,11000,1265761.0,4123.0,0.844615,23.39,942.38,46.2,150,50,40,8.0
5,Groundnut,1997,Kharif,Punjab,8000.0,8000,761360.0,2480.0,1.000000,23.39,942.38,46.2,150,50,40,8.0
6,Maize,1997,Kharif,Punjab,165000.0,345000,15703050.0,51150.0,2.102727,23.39,942.38,46.2,150,50,40,8.0
7,Masoor,1997,Rabi,Punjab,5200.0,3400,494884.0,1612.0,0.726250,23.39,942.38,46.2,150,50,40,8.0
8,Moong(Green Gram),1997,Whole Year,Punjab,49300.0,31700,4691881.0,15283.0,0.601667,23.39,942.38,46.2,150,50,40,8.0
9,Rapeseed &Mustard,1997,Rabi,Punjab,72000.0,63000,6852240.0,22320.0,0.925882,23.39,942.38,46.2,150,50,40,8.0


In [13]:
# CELL 4: Feature Engineering for the 9-Crop Model

# 1. Map dataset crop names to our standard 9 crops
# Datasets often have slight spelling variations, so we catch them here.
CROP_MAPPING = {
    'Rice': 'Rice',
    'Wheat': 'Wheat',
    'Maize': 'Maize',
    'Cotton(Lint)': 'Cotton(Lint)',
    'Sugarcane': 'Sugarcane',
    'Rapeseed &Mustard': 'Mustard',
    'Mustard': 'Mustard',
    'Bajra': 'Bajra',
    'Barley': 'Barley',
    'Arhar/Tur': 'Arhar/Tur',
    'Tur': 'Arhar/Tur'
}

# Apply the standard naming and drop crops we don't need
df_merged['Crop_Standard'] = df_merged['Crop'].map(CROP_MAPPING)
df_filtered = df_merged.dropna(subset=['Crop_Standard']).copy()

# Remove any rows with zero production or area
df_filtered = df_filtered[(df_filtered['Production'] > 0) & (df_filtered['Area'] > 0)].copy()

# 2. Agronomic Parameters for all 9 Crops (PAU / HAU / ICAR Standards)
RPR_MAP = {
    'Rice': 1.6,         # Paddy straw
    'Wheat': 1.4,        # Wheat turi
    'Maize': 1.8,        # Stalks and cobs
    'Cotton(Lint)': 2.8, # Woody stalks
    'Sugarcane': 0.35,   # Cane trash/tops
    'Mustard': 1.5,      # Woody stalks
    'Bajra': 2.0,        # Pearl millet stalks
    'Barley': 1.3,       # Barley straw
    'Arhar/Tur': 3.0     # Woody pigeon pea bushes
}

BASE_DAYS_MAP = {
    'Rice': 120,
    'Wheat': 142,
    'Maize': 95,
    'Cotton(Lint)': 155,
    'Sugarcane': 330,
    'Mustard': 135,
    'Bajra': 85,
    'Barley': 125,
    'Arhar/Tur': 160
}

# 3. Target 2: Calculate Biomass Tons
# Convert Hectares to Acres for the UI (1 Hectare = 2.47105 Acres)
df_filtered['farm_area_acres'] = np.round(df_filtered['Area'] * 2.47105, 2)

df_filtered['rpr'] = df_filtered['Crop_Standard'].map(RPR_MAP)
df_filtered['biomass_tons'] = np.round(df_filtered['Production'] * df_filtered['rpr'], 2)

# 4. Target 1: Calculate Harvest Days Remaining
# Adjust the baseline crop cycle using actual temperature and rainfall data
temp_effect = -0.55 * (df_filtered['Temperature'] - df_filtered['Temperature'].mean())
rain_effect = 0.008 * (df_filtered['Rainfall'] - df_filtered['Rainfall'].mean())

df_filtered['base_days'] = df_filtered['Crop_Standard'].map(BASE_DAYS_MAP)
df_filtered['total_growth_days'] = df_filtered['base_days'] + temp_effect + rain_effect

# Simulate the farmer checking the app mid-season (55% to 95% through the cycle)
np.random.seed(42)
df_filtered['days_since_sowing'] = np.round(df_filtered['total_growth_days'] * np.random.uniform(0.55, 0.95, len(df_filtered)))
df_filtered['harvest_days_remaining'] = np.maximum(1, np.round(df_filtered['total_growth_days'] - df_filtered['days_since_sowing'])).astype(int)

# 5. Simulate Satellite Phenology (NDVI profile)
# NDVI is highest mid-season and drops as crops ripen and dry out for harvest
maturity_pct = df_filtered['days_since_sowing'] / df_filtered['total_growth_days']
base_ndvi = np.where(maturity_pct < 0.75, 0.78, 0.78 - (maturity_pct - 0.75) * 1.6)
df_filtered['ndvi'] = np.clip(base_ndvi + np.random.normal(0, 0.03, len(df_filtered)), 0.25, 0.88).round(3)
df_filtered['evi'] = np.clip(df_filtered['ndvi'] * 0.70 + np.random.normal(0, 0.02, len(df_filtered)), 0.15, 0.65).round(3)

# 6. Ensure Location is set
df_filtered['location'] = df_filtered['State']

print(f"✅ Final 9-Crop Dataset Ready! Total records: {len(df_filtered)}")
display(df_filtered[['Crop_Standard', 'location', 'farm_area_acres', 'days_since_sowing', 'Temperature', 'harvest_days_remaining', 'biomass_tons']].head(10))

✅ Final 9-Crop Dataset Ready! Total records: 403


,Crop_Standard,location,farm_area_acres,days_since_sowing,Temperature,harvest_days_remaining,biomass_tons
0,Arhar/Tur,Punjab,27181.55,112.0,23.39,48,24600.0
1,Bajra,Punjab,19768.40,79.0,23.39,6,16000.0
2,Barley,Punjab,91428.85,105.0,23.39,20,144300.0
3,Cotton(Lint),Punjab,1789040.20,122.0,23.39,33,2623600.0
6,Maize,Punjab,407723.25,58.0,23.39,37,621000.0
9,Mustard,Punjab,177915.60,83.0,23.39,52,94500.0
10,Rice,Punjab,5636465.05,69.0,23.39,51,12646400.0
12,Sugarcane,Punjab,311352.30,296.0,23.39,34,2502500.0
14,Wheat,Punjab,8154465.00,112.0,23.39,30,17801000.0
15,Arhar/Tur,Haryana,35205.05,133.0,24.04,27,40500.0


In [17]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 19.6 MB/s eta 0:00:03
   --- ------------------------------------ 4.5/48.9 MB 19.0 MB/s eta 0:00:03
   ----- ---------------------------------- 7.1/48.9 MB 16.0 MB/s eta 0:00:03
   ------- -------------------------------- 9.7/48.9 MB 15.0 MB/s eta 0:00:03
   --------- ------------------------------ 12.1/48.9 MB 14.2 MB/s eta 0:00:03
   ------------ --------------------------- 14.9/48.9 MB 13.9 MB/s eta 0:00:03
   -------------- ------------------------- 17.6/48.9 MB 13.6 MB/s eta 0:00:03
   ---------------- ----------------------- 19.9/48.9 MB 13.5 MB/s eta 0:00:03
   ------------------ --------------------- 22.8/48.9 MB 13.4 MB/s eta 0:00:02
   -------------------- ------------------- 25.2/48.9 MB 13.4 MB/s eta 0:00:02
   ---------------------- ----------------- 27.8/48.9 MB 13.2 MB/s eta 0:00:02
   ------------------------ --------------- 30.4/48.9 MB 13.2 MB/


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# CELL 5: The Model Showdown (Ridge vs. Random Forest vs. XGBoost)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
# 1. Define Features and Targets
feature_cols = [
    'Crop_Standard', 'location', 'farm_area_acres', 'days_since_sowing',
    'Temperature', 'Rainfall', 'Humidity', 'ndvi', 'evi', 
    'N', 'P', 'K', 'pH'
]

categorical_cols = ['Crop_Standard', 'location']
numeric_cols = [
    'farm_area_acres', 'days_since_sowing', 'Temperature', 'Rainfall', 
    'Humidity', 'ndvi', 'evi', 'N', 'P', 'K', 'pH'
]

# Set up the preprocessing pipeline WITH StandardScaler for fair comparison
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', numeric_transformer, numeric_cols)
])

X = df_filtered[feature_cols]
y_harvest = df_filtered['harvest_days_remaining']
y_biomass = df_filtered['biomass_tons']

# 2. Define the Models to Compare
models = {
    "Ridge (Baseline)": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=16, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=150, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1)
}

results_harvest = []
results_biomass = []

# --- Split the Data ---
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X, y_harvest, test_size=0.2, random_state=42)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X, y_biomass, test_size=0.2, random_state=42)

# 3. Train and Evaluate Each Model
print("Training models and calculating metrics. Please wait...\n")

for name, model in models.items():
    # Train Harvest Model
    h_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    h_pipeline.fit(X_train_h, y_train_h)
    h_preds = h_pipeline.predict(X_test_h)
    h_mae = mean_absolute_error(y_test_h, h_preds)
    h_r2 = r2_score(y_test_h, h_preds)
    
    results_harvest.append({"Model": name, "MAE (Days)": round(h_mae, 3), "R² Score": round(h_r2, 4)})
    
    # Train Biomass Model
    b_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    b_pipeline.fit(X_train_b, y_train_b)
    b_preds = b_pipeline.predict(X_test_b)
    b_mae = mean_absolute_error(y_test_b, b_preds)
    b_r2 = r2_score(y_test_b, b_preds)
    
    results_biomass.append({"Model": name, "MAE (Tons)": round(b_mae, 3), "R² Score": round(b_r2, 4)})

# 4. Display the Results nicely
print("🏆 === HARVEST PREDICTION SHOWDOWN ===")
display(pd.DataFrame(results_harvest).sort_values(by="MAE (Days)"))

print("\n🏆 === BIOMASS ESTIMATION SHOWDOWN ===")
display(pd.DataFrame(results_biomass).sort_values(by="MAE (Tons)"))

Training models and calculating metrics. Please wait...

🏆 === HARVEST PREDICTION SHOWDOWN ===


,Model,MAE (Days),R² Score
2,XGBoost,3.902,0.9174
0,Ridge (Baseline),3.998,0.9203
1,Random Forest,4.007,0.9160



🏆 === BIOMASS ESTIMATION SHOWDOWN ===


,Model,MAE (Tons),R² Score
2,XGBoost,429820.559,0.9773
1,Random Forest,466525.601,0.9775
0,Ridge (Baseline),669038.808,0.9778


In [19]:
# CELL 6: Fix Scale, Train Champion XGBoost, and Save Models
import os
import joblib
import numpy as np
from xgboost import XGBRegressor

# 1. THE FIX: Convert district-level data to farmer-level data
# Calculate the actual yield per acre for that specific district/year
df_filtered['yield_per_acre'] = df_filtered['Production'] / (df_filtered['Area'] * 2.47105)

# Simulate realistic farmer field sizes (e.g., 2 to 50 acres) instead of whole cities
np.random.seed(42)
df_filtered['farm_area_acres'] = np.random.uniform(2.0, 50.0, len(df_filtered)).round(1)

# Recalculate realistic farmer biomass: Area * Yield * Crop Residue Ratio
df_filtered['biomass_tons'] = np.round(df_filtered['farm_area_acres'] * df_filtered['yield_per_acre'] * df_filtered['rpr'], 2)

# Update our features and targets with the fixed data
X = df_filtered[feature_cols]
y_harvest = df_filtered['harvest_days_remaining']
y_biomass = df_filtered['biomass_tons']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X, y_harvest, test_size=0.2, random_state=42)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X, y_biomass, test_size=0.2, random_state=42)

# 2. TRAIN THE CHAMPION
print("Training the winning XGBoost models on realistic farm data...")

harvest_champion = Pipeline([
    ('preprocessor', preprocessor), 
    ('regressor', XGBRegressor(n_estimators=150, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1))
])
harvest_champion.fit(X_train_h, y_train_h)

biomass_champion = Pipeline([
    ('preprocessor', preprocessor), 
    ('regressor', XGBRegressor(n_estimators=150, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1))
])
biomass_champion.fit(X_train_b, y_train_b)

# Quick check to ensure the MAE is now normal!
b_preds = biomass_champion.predict(X_test_b)
print(f"Fixed Biomass MAE: {mean_absolute_error(y_test_b, b_preds):.2f} tons (Much better!)")

# 3. SAVE THE MODELS FOR FASTAPI
# Make sure the artifacts folder exists in your backend
ARTIFACTS_DIR = os.path.abspath("../../artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

harvest_path = os.path.join(ARTIFACTS_DIR, "harvest_model.joblib")
biomass_path = os.path.join(ARTIFACTS_DIR, "biomass_model.joblib")

joblib.dump(harvest_champion, harvest_path)
joblib.dump(biomass_champion, biomass_path)

print(f"\n✅ Models successfully saved!")
print(f"Ready for backend at: {ARTIFACTS_DIR}")

Training the winning XGBoost models on realistic farm data...
Fixed Biomass MAE: 26.15 tons (Much better!)

✅ Models successfully saved!
Ready for backend at: c:\Users\Aryan Sharma\Desktop\artifacts
